In [0]:
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

In [0]:
SOURCE_TABLE = "h2.silver.model_training_day"
LABEL_COL = "high_usage_next_24h"
TIME_COL = "event_time_utc"
ZONE_COL ="zone"
SPLIT_TS = "2021-01-01 00:00:00"

# df = spark.table(SOURCE_TABLE).filter(F.col(ZONE_COL)=='NL')
# df = df.withColumn(LABEL_COL, F.col(LABEL_COL).cast('int')).filter(F.col(LABEL_COL).isNotNull())

# train = df.filter(F.col("event_time_utc") < F.to_timestamp(F.lit(SPLIT_TS)))
# test = df.filter(F.col("event_time_utc") >= F.to_timestamp(F.lit(SPLIT_TS)))

test = spark.table("h2.silver.model_test").filter(F.col(ZONE_COL)=='NL')
train = spark.table("h2.silver.model_training").filter(F.col(ZONE_COL)=='NL')

In [0]:
exclude_exact = {LABEL_COL, TIME_COL, ZONE_COL, "event_date", "season"}
exclude_prefixes = {"next_","load_", "feature_"}
numeric_types = ('double', 'float', 'int', 'bigint', 'long', 'short', 'byte')

feature_cols = []
for f in train.schema.fields:
  name = f.name
  dtype = f.dataType.simpleString().lower()
  if name in exclude_exact:
      continue
  if any(name.startswith(p) for p in exclude_prefixes):
      continue
  if any(t in dtype for t in numeric_types):
      feature_cols.append(name)
if not feature_cols:
    raise ValueError("No numeric columns found")
print("Feature count", len(feature_cols))
print("Sample features", feature_cols[:20])

In [0]:
#Class weights

counts = train.groupBy(LABEL_COL).count().collect()
count_map = {int(r[LABEL_COL]):int(r['count']) for r in counts}
n0 = count_map.get(0,0)
n1 = count_map.get(1,0)

if n0 == 0 or n1 == 1:
    raise ValueError("Train split has only one class: {count_map}")
total = n0 + n1 
w0= total/(2.0 * n0)
w1= total/(2.0 * n1)
# print("Class weights", w0, w1)

train = train.withColumn("class_weight", F.when(F.col(LABEL_COL)==1, F.lit(w1)).otherwise(F.lit(w0)))
test = test.withColumn("class_weight", F.lit(1.0))

print("Class counts(train):", count_map,"weights:",{'0':w0, "1":w1})




In [0]:
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

lr = LogisticRegression(
    featuresCol="features",
    labelCol=LABEL_COL,
    weightCol="class_weight",
    maxIter=100,
    regParam=0.01,
    elasticNetParam=0.0,
    standardization=True,
    threshold=0.5,
    fitIntercept=True
)

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol=LABEL_COL,
    weightCol="class_weight",
    maxDepth=12,
    numTrees=300,
    seed= 42
)

pipeline_lr = Pipeline(stages=[assembler, lr])
pipeline_rf = Pipeline(stages=[assembler, rf])

model_lr = pipeline_lr.fit(train)
model_rf = pipeline_rf.fit(train)

pred_lr = model_lr.transform(test)
pred_rf = model_rf.transform(test)


In [0]:
#Evaluation
auc_eval = BinaryClassificationEvaluator(labelCol=LABEL_COL,
                                         rawPredictionCol="rawPrediction", metricName="areaUnderROC")
f1_eval = MulticlassClassificationEvaluator(labelCol=LABEL_COL,
                                            predictionCol="prediction", metricName="f1")
p_eval = MulticlassClassificationEvaluator(labelCol=LABEL_COL,
                                           predictionCol="prediction",
                                           metricName="weightedPrecision")
r_eval = MulticlassClassificationEvaluator(labelCol=LABEL_COL,
                                           predictionCol="prediction",
                                           metricName="weightedRecall")

auc_lr = auc_eval.evaluate(pred_lr)
auc_rf = auc_eval.evaluate(pred_rf)
f1_lr = f1_eval.evaluate(pred_lr)
f1_rf = f1_eval.evaluate(pred_rf)
p_lr = p_eval.evaluate(pred_lr)
p_rf = p_eval.evaluate(pred_rf)
r_lr = r_eval.evaluate(pred_lr)
r_rf = r_eval.evaluate(pred_rf)

metrics =[
    ("logistic_regression",float(auc_lr),
     float(f1_lr),float(p_lr),float(r_lr)),
    ("random_forest",float(auc_rf),
     float(f1_rf),float(p_rf),float(r_rf))
]

metrics_df = spark.createDataFrame(metrics,["model_name", "auc", "f1", "precision", "recall"])\
    .withColumn("evaluated_at_utc",F.current_timestamp())
display(metrics_df.orderBy(F.desc("auc"), F.desc("f1")))






In [0]:
# 1) Train/Test overlap check
overlap = train.select("event_time_utc","zone").intersect(
    test.select("event_time_utc","zone")
).count()
print("overlap rows:", overlap)
# 2) Leakage columns check
print([c for c in feature_cols if c.startswith(("next_", "lead_", "future_", "p75_"))])
# 3) Confirm eval really on test
print("train rows:", train.count(), "test rows:", test.count())
# 4) Hard check: remove strongest direct signal and re-test
reduced_features = [c for c in feature_cols if c not in ["avg_actual_total_load_mw", "day_ahead_total_load_forecast_mw"]]
print("reduced feature count:", len(reduced_features))


In [0]:
from pyspark.ml.functions import vector_to_array
from pyspark.sql import functions as F
winner = metrics_df.orderBy(F.desc("auc"), F.desc("f1")).first()["model_name"]
print("Winner:", winner)
winner_pred = (
    pred_lr if winner == "logistic_regression" else pred_rf
).select(
    TIME_COL,
    ZONE_COL,
    F.col(LABEL_COL).alias("actual_label"),
    F.col("prediction").cast("int").alias("predicted_label"),
    vector_to_array(F.col("probability"))[1].alias("score_positive")  # fixed
).withColumn(
    "model_name", F.lit(winner)
).withColumn(
    "scored_at_utc", F.current_timestamp()
)
metrics_df.write.mode("overwrite").format("delta").saveAsTable("h2.gold.day6_model_metrics")
winner_pred.write.mode("overwrite").format("delta").saveAsTable("h2.gold.day6_winner_predictions")
print("Saved -> h2.gold.day6_model_metrics")
print("Saved -> h2.gold.day6_winner_predictions")

In [0]:
from pyspark.sql import functions as F
import pandas as pd
import numpy as np
# If needed in your cluster:
# %pip install statsmodels
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error
# -----------------------------------
# 1) Load and prepare time series data
# -----------------------------------
SOURCE_TABLE = "h2.silver.model_training_day"   # change if needed
TIME_COL = "event_time_utc"
TARGET_COL = "avg_actual_total_load_mw"
ZONE_COL = "zone"
df = (
    spark.table(SOURCE_TABLE)
    .filter(F.col(ZONE_COL) == "NL")
    .select(TIME_COL, TARGET_COL)
    .filter(F.col(TARGET_COL).isNotNull())
    .orderBy(TIME_COL)
)

In [0]:
pdf = df.toPandas()
pdf[TIME_COL] = pd.to_datetime(pdf[TIME_COL], utc=True)
pdf = pdf.sort_values(TIME_COL).drop_duplicates(subset=[TIME_COL])
# Set hourly index and fill missing timestamps
pdf = pdf.set_index(TIME_COL).asfreq("h")
pdf[TARGET_COL] = pdf[TARGET_COL].interpolate(method="time").ffill().bfill()

In [0]:

# # -----------------------------------
# # 2) Train/test split (2020 -> 2021)
# # -----------------------------------
# train = pdf.loc[pdf.index < "2021-01-01"]
# test  = pdf.loc[pdf.index >= "2021-01-01"]
# y_train = train[TARGET_COL]
# y_test = test[TARGET_COL]
# print("Train rows:", len(y_train), "Test rows:", len(y_test))
# # -----------------------------------
# # 3) Fit SARIMAX (ARIMA-like)
# # Try one stable baseline first
# # -----------------------------------
# # order=(p,d,q), seasonal_order=(P,D,Q,s)
# # s=24 for hourly daily seasonality
# model = SARIMAX(
#     y_train,
#     order=(2, 1, 2),
#     seasonal_order=(1, 0, 1, 24),
#     enforce_stationarity=False,
#     enforce_invertibility=False
# )
# fit = model.fit(disp=False)
# # -----------------------------------
# # 4) Forecast full test horizon
# # -----------------------------------
# forecast = fit.get_forecast(steps=len(y_test)).predicted_mean
# forecast.index = y_test.index
# # -----------------------------------
# # 5) Metrics
# # -----------------------------------
# mae = mean_absolute_error(y_test, forecast)
# rmse = mean_squared_error(y_test, forecast, squared=False)
# mape = np.mean(np.abs((y_test - forecast) / np.clip(np.abs(y_test), 1e-6, None))) * 100
# print(f"MAE : {mae:.4f}")
# print(f"RMSE: {rmse:.4f}")
# print(f"MAPE: {mape:.2f}%")
# # -----------------------------------
# # 6) Save forecast results to Gold
# # -----------------------------------
# pred_pd = pd.DataFrame({
#     "event_time_utc": forecast.index,
#     "zone": "NL",
#     "actual_load_mw": y_test.values,
#     "predicted_load_mw": forecast.values
# })
# pred_sdf = spark.createDataFrame(pred_pd) \
#     .withColumn("model_name", F.lit("sarimax_arima")) \
#     .withColumn("scored_at_utc", F.current_timestamp())
# metrics_pd = pd.DataFrame([{
#     "model_name": "sarimax_arima",
#     "mae": float(mae),
#     "rmse": float(rmse),
#     "mape": float(mape)
# }])
# metrics_sdf = spark.createDataFrame(metrics_pd).withColumn("evaluated_at_utc", F.current_timestamp())
# pred_sdf.write.mode("overwrite").format("delta").saveAsTable("h2.gold.day6_arima_predictions")
# metrics_sdf.write.mode("overwrite").format("delta").saveAsTable("h2.gold.day6_arima_metrics")
# print("Saved -> h2.gold.day6_arima_predictions")
# print("Saved -> h2.gold.day6_arima_metrics")
# # Optional: if you still want binary high-usage output from ARIMA forecast:
# # Build binary proxy from predicted load using train p75 threshold
# p75 = float(np.percentile(y_train.values, 75))
# pred_flag = pred_sdf.withColumn("high_usage_pred", F.when(F.col("predicted_load_mw") >= F.lit(p75), F.lit(1)).otherwise(F.lit(0)))
# pred_flag.write.mode("overwrite").format("delta").saveAsTable("h2.gold.day6_arima_predictions_flagged")

In [0]:
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.functions import vector_to_array
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

In [0]:
TRAIN_TABLE = "h2.silver.model_training"       # your 2020 train table
TEST_TABLE = "h2.silver.model_test"            # your 2021 test table
FULL_TABLE = "h2.silver.model_training_day"    # full engineered table (for ARIMA timeline)
ZONE = "NL"

TIME_COL = "event_time_utc"
ZONE_COL = "zone"
LABEL_COL = "high_usage_next_24h"
TARGET_NUMERIC = "avg_actual_total_load_mw"

OUT_LEADERBOARD = "h2.gold.day6_model_leaderboard"
OUT_CLASS_PRED = "h2.gold.day6_classifier_predictions"
OUT_ARIMA_PRED = "h2.gold.day6_arima_predictions"
OUT_ARIMA_NUM = "h2.gold.day6_arima_numeric_metrics"

In [0]:
train_df = spark.table(TRAIN_TABLE).filter(F.col(ZONE_COL) == ZONE)
test_df  = spark.table(TEST_TABLE).filter(F.col(ZONE_COL) == ZONE)

train_df = train_df.withColumn(LABEL_COL, F.col(LABEL_COL).cast("int")).filter(F.col(LABEL_COL).isNotNull())
test_df  = test_df.withColumn(LABEL_COL, F.col(LABEL_COL).cast("int")).filter(F.col(LABEL_COL).isNotNull())

print("train rows:", train_df.count())
print("test rows :", test_df.count())

In [0]:
exclude_exact = {LABEL_COL, TIME_COL, ZONE_COL, "event_date", "season"}
exclude_prefixes = ("next_", "lead_", "future_", "p75_")
numeric_types = ("double", "float", "int", "bigint", "smallint", "tinyint", "decimal")
feature_cols = []
for f in train_df.schema.fields:
    n = f.name
    t = f.dataType.simpleString().lower()
    if n in exclude_exact:
        continue
    if any(n.startswith(p) for p in exclude_prefixes):
        continue
    if any(x in t for x in numeric_types):
        feature_cols.append(n)
if not feature_cols:
    raise ValueError("No usable numeric features after leakage filters.")
print("feature count:", len(feature_cols))
print("sample features:", feature_cols[:20])

In [0]:
counts = train_df.groupBy(LABEL_COL).count().collect()
count_map = {int(r[LABEL_COL]): int(r["count"]) for r in counts}
n0 = count_map.get(0, 0)
n1 = count_map.get(1, 0)

if n0 == 0 or n1 == 0:
    raise ValueError(f"Train has single class: {count_map}")

total = n0 + n1
w0 = total / (2.0 * n0)
w1 = total / (2.0 * n1)

train_w = train_df.withColumn("class_weight", F.when(F.col(LABEL_COL) == 1, F.lit(w1)).otherwise(F.lit(w0)))
test_w  = test_df.withColumn("class_weight", F.lit(1.0))
cw1 = total / (2.0 * n1)
train_w = train_df.withColumn("class_weight", F.when(F.col(LABEL_COL) == 1, F.lit(w1)).otherwise(F.lit(w0)))
test_w  = test_df.withColumn("class_weight", F.lit(1.0))

print("class counts:", count_map, "weights:", {"0": w0, "1": w1})

In [0]:
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features", handleInvalid="keep")
lr = LogisticRegression(
    featuresCol="features",
    labelCol=LABEL_COL,
    weightCol="class_weight",
    maxIter=100,
    regParam=0.01,
    elasticNetParam=0.0
)
gbt = GBTClassifier(
    featuresCol="features",
    labelCol=LABEL_COL,
    maxIter=120,
    maxDepth=5,
    stepSize=0.05,
    subsamplingRate=0.8,
    seed=42
)
model_lr = Pipeline(stages=[assembler, lr]).fit(train_w)
model_gbt = Pipeline(stages=[assembler, gbt]).fit(train_w)
pred_lr = model_lr.transform(test_w)
pred_gbt = model_gbt.transform(test_w)

In [0]:
auc_eval = BinaryClassificationEvaluator(labelCol=LABEL_COL, rawPredictionCol="rawPrediction", metricName="areaUnderROC")
pr_eval  = BinaryClassificationEvaluator(labelCol=LABEL_COL, rawPredictionCol="rawPrediction", metricName="areaUnderPR")
f1_eval  = MulticlassClassificationEvaluator(labelCol=LABEL_COL, predictionCol="prediction", metricName="f1")
p_eval   = MulticlassClassificationEvaluator(labelCol=LABEL_COL, predictionCol="prediction", metricName="weightedPrecision")
r_eval   = MulticlassClassificationEvaluator(labelCol=LABEL_COL, predictionCol="prediction", metricName="weightedRecall")
def cls_metrics(pred_df, name):
    return (
        name,
        float(auc_eval.evaluate(pred_df)),
        float(pr_eval.evaluate(pred_df)),
        float(f1_eval.evaluate(pred_df)),
        float(p_eval.evaluate(pred_df)),
        float(r_eval.evaluate(pred_df)),
    )
cls_rows = [
    cls_metrics(pred_lr, "logistic_regression"),
    cls_metrics(pred_gbt, "gbt"),
]
cls_metrics_df = spark.createDataFrame(
    cls_rows,
    ["model_name", "auc", "pr_auc", "f1", "precision", "recall"]
).withColumn("task_type", F.lit("classification"))
display(cls_metrics_df.orderBy(F.desc("f1"), F.desc("auc")))

In [0]:
winner_cls = cls_metrics_df.orderBy(F.desc("f1"), F.desc("auc")).first()["model_name"]
print("classification winner:", winner_cls)
pred_win = pred_lr if winner_cls == "logistic_regression" else pred_gbt
winner_pred = (
    pred_win
    .select(
        TIME_COL,
        ZONE_COL,
        F.col(LABEL_COL).alias("actual_label"),
        F.col("prediction").cast("int").alias("predicted_label"),
        vector_to_array(F.col("probability"))[1].alias("score_positive")
    )
    .withColumn("model_name", F.lit(winner_cls))
    .withColumn("task_type", F.lit("classification"))
    .withColumn("scored_at_utc", F.current_timestamp())
)
winner_pred.write.mode("overwrite").format("delta").saveAsTable(OUT_CLASS_PRED)
print(f"Saved -> {OUT_CLASS_PRED}")

###ARIMA

In [0]:
full_sdf = (
    spark.table(FULL_TABLE)
    .filter(F.col(ZONE_COL) == ZONE)
    .select(TIME_COL, TARGET_NUMERIC)
    .filter(F.col(TARGET_NUMERIC).isNotNull())
    .orderBy(TIME_COL)
)
pdf = full_sdf.toPandas()
pdf[TIME_COL] = pd.to_datetime(pdf[TIME_COL], utc=True)
pdf = pdf.sort_values(TIME_COL).drop_duplicates(subset=[TIME_COL])
pdf = pdf.set_index(TIME_COL).asfreq("h")
pdf[TARGET_NUMERIC] = pdf[TARGET_NUMERIC].interpolate(method="time").ffill().bfill()

y_train = pdf.loc[pdf.index < "2021-01-01", TARGET_NUMERIC]
y_test = pdf.loc[pdf.index >= "2021-01-01", TARGET_NUMERIC]
print("ARIMA train:", len(y_train), "ARIMA test:", len(y_test))
arima = SARIMAX(
    y_train,
    order=(2, 1, 2),
    seasonal_order=(1, 0, 1, 24),
    enforce_stationarity=False,
    enforce_invertibility=False
)
arima_fit = arima.fit(disp=False)
forecast = arima_fit.get_forecast(steps=len(y_test)).predicted_mean
forecast.index = y_test.index

In [0]:
# ---------------------------
# 9) ARIMA numeric metrics
# ---------------------------
mae = float(mean_absolute_error(y_test, forecast))
rmse = float(np.sqrt(mean_squared_error(y_test, forecast)))
mape = float(np.mean(np.abs((y_test - forecast) / np.clip(np.abs(y_test), 1e-6, None))) * 100)
print("ARIMA MAE :", mae)
print("ARIMA RMSE:", rmse)
print("ARIMA MAPE:", mape)
arima_num_df = spark.createDataFrame(
    [("arima", mae, rmse, mape)],
    ["model_name", "mae", "rmse", "mape"]
).withColumn("task_type", F.lit("forecasting_numeric")) \
 .withColumn("evaluated_at_utc", F.current_timestamp())
arima_num_df.write.mode("overwrite").format("delta").saveAsTable(OUT_ARIMA_NUM)
print(f"Saved -> {OUT_ARIMA_NUM}")

In [0]:
# ---------------------------
# 10) ARIMA -> binary proxy for same leaderboard (fair compare)
# ---------------------------
# Use train p75 threshold (same spirit as your Day 5 label)
p75 = float(np.percentile(y_train.values, 75))
arima_pd = pd.DataFrame({
    "event_time_utc": forecast.index,
    "actual_value": y_test.values,
    "predicted_value": forecast.values
})
arima_pd["actual_label"] = (arima_pd["actual_value"] >= p75).astype(int)
arima_pd["predicted_label"] = (arima_pd["predicted_value"] >= p75).astype(int)

# smooth score for AUC/PR-AUC
scale = max(1.0, float(np.std(y_train.values)))
arima_pd["score_positive"] = 1.0 / (1.0 + np.exp(-(arima_pd["predicted_value"] - p75) / scale))
arima_auc = float(roc_auc_score(arima_pd["actual_label"], arima_pd["score_positive"]))
arima_prauc = float(average_precision_score(arima_pd["actual_label"], arima_pd["score_positive"]))
arima_f1 = float(f1_score(arima_pd["actual_label"], arima_pd["predicted_label"], zero_division=0))
arima_prec = float(precision_score(arima_pd["actual_label"], arima_pd["predicted_label"], zero_division=0))
arima_rec = float(recall_score(arima_pd["actual_label"], arima_pd["predicted_label"], zero_division=0))
arima_cls_df = spark.createDataFrame(
    [("arima_thresholded", arima_auc, arima_prauc, arima_f1, arima_prec, arima_rec)],
    ["model_name", "auc", "pr_auc", "f1", "precision", "recall"]
).withColumn("task_type", F.lit("forecasting_binary_proxy"))
display(arima_cls_df)

In [0]:
# ---------------------------
# 11) Unified leaderboard (all together)
# ---------------------------
leaderboard = (
    cls_metrics_df
    .unionByName(arima_cls_df)
    .withColumn("evaluated_at_utc", F.current_timestamp())
)
display(leaderboard.orderBy(F.desc("f1"), F.desc("pr_auc"), F.desc("auc")))
leaderboard.write.mode("overwrite").format("delta").saveAsTable(OUT_LEADERBOARD)
print(f"Saved -> {OUT_LEADERBOARD}")
# ---------------------------
# 12) Save ARIMA prediction table
# ---------------------------
arima_pred_sdf = spark.createDataFrame(arima_pd) \
    .withColumn("zone", F.lit(ZONE)) \
    .withColumn("model_name", F.lit("arima")) \
    .withColumn("scored_at_utc", F.current_timestamp())
arima_pred_sdf.write.mode("overwrite").format("delta").saveAsTable(OUT_ARIMA_PRED)
print(f"Saved -> {OUT_ARIMA_PRED}")

In [0]:
import mlflow
from pyspark.sql import functions as F
mlflow.set_experiment("/Shared/h2_day7_experiment")
m = spark.table("h2.gold.day6_model_leaderboard").orderBy(F.desc("f1"), F.desc("auc")).first()
with mlflow.start_run(run_name=f"day7_{m['model_name']}"):
    mlflow.log_param("model_name", m["model_name"])
    mlflow.log_param("zone", "NL")
    mlflow.log_param("label", "high_usage_next_24h_proxy")
    mlflow.log_param("task_type", m["task_type"])
    mlflow.log_metric("auc", float(m["auc"]))
    mlflow.log_metric("pr_auc", float(m["pr_auc"]))
    mlflow.log_metric("f1", float(m["f1"]))
    mlflow.log_metric("precision", float(m["precision"]))
    mlflow.log_metric("recall", float(m["recall"]))